# Phase 4 — MLflow Experiment Tracking
## Credit Scoring System

**Prerequisites:**
- MLflow UI running at http://127.0.0.1:5000
- model_table.csv in outputs/ folder

**Run in terminal before this notebook:**
```bash
cd Healthcare
mlflow ui
```

In [2]:
import mlflow
import pandas as pd
import os
import joblib
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import roc_auc_score, average_precision_score
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print(f"MLflow Version: {mlflow.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

MLflow Version: 3.17.0
Tracking URI: sqlite:///../mlflow.db


In [3]:
# Set the experiment
mlflow.set_experiment("credit-score-prediction")
print("Experiment Created!")

2026/10/18 11:15:40 INFO mlflow.tracking.fluent: Experiment with name 'credit-score-prediction' does not exist. Creating a new experiment.


Experiment Created!


In [4]:
# Step 1) Load save models
xgb_score_model = joblib.load("../models/score_model.joblib")
print("Model loaded!")
print(f"score_model: {type(xgb_score_model)}")

Model loaded!
score_model: <class 'sklearn.pipeline.Pipeline'>


In [5]:
# Step 2) Load the dataset
df = pd.read_csv("../outputs/model_table.csv")
print("Data Loaded")
print(f"Dataset shape: {df.shape}")
df.head()

Data Loaded
Dataset shape: (150000, 14)


,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,total-number_of_lates,log_transformed-montly-income
0,1,1,0.766127,45.0,2,0.802982,9120.0,13,0,6,0,2.0,2,9.118335
1,2,0,0.957151,40.0,0,0.121876,2600.0,4,0,0,0,1.0,0,7.863651
2,3,0,0.658180,38.0,1,0.085113,3042.0,2,1,0,0,0.0,2,8.020599
3,4,0,0.233810,30.0,0,0.036050,3300.0,5,0,0,0,0.0,0,8.101981
4,5,0,0.907239,49.0,1,0.024926,63588.0,7,0,1,0,0.0,1,11.060196


In [6]:
# Step 3) Load feature_schema
with open("../outputs/feature_schema.json", "r") as f:
    feature_schema = json.load(f)

score_features = feature_schema["score_model_features"]
score_target = feature_schema["score_target"]

print("Feature schema loaded!")
print(f"Score features: {len(score_features)}")

Feature schema loaded!
Score features: 12


In [7]:
# Step) Create separate dataset
score_df = df.copy()

In [8]:
# Step 5) Train/Test Split
split_idx = int(len(score_df)*0.8)

score_train = score_df.iloc[:split_idx].copy()
score_test = score_df.iloc[split_idx:].copy()

X_train_score = score_train[score_features]
X_test_score = score_test[score_features]

y_train_score = score_train[score_target]
y_test_score = score_test[score_target]

print("Train shape: ", X_train_score.shape)
print("Test shape: ", X_test_score.shape)

Train shape:  (120000, 12)
Test shape:  (30000, 12)


In [20]:
# Step 6) Log in MLflow
with mlflow.start_run(run_name="XGBoost-Score") as run:
    # Log the parameters of the model
    mlflow.log_params({
        "model": "XGBClassifier",
        "objective": 'binary:logistic',
        "n_estimators": 400,
        "max_depth": 6,
        "learning_rate": 0.014831660073622589,
        "subsample": 0.815584046804639,
        "colsample_bytree": 0.763165386967939,
        "reg_alpha": 8.750244277935877,
        "reg_lamda": 7.195684092208847,
        "scale_pos_weight": 1.8630641212756873

    })

    # Prediction
    pred_score = xgb_score_model.predict_proba(X_test_score)[:, 1]

    # Metrics
    roc_auc = roc_auc_score(y_test_score, pred_score)
    average_precision = average_precision_score(y_test_score, pred_score)

    # Metrics
    mlflow.log_metric("ROC-AUC score", roc_auc)
    mlflow.log_metric("Average Precision", average_precision)

    # Log Model
    mlflow.sklearn.log_model(
    sk_model=xgb_score_model,
    name="model",
    skops_trusted_types=[
        "numpy.dtype",
        "xgboost.core.Booster",
        "xgboost.sklearn.XGBClassifier"
        ]
    )

    # Fetching run id
    score_run_id = run.info.run_id

    print("XGBoost Score Model Logged!")
    print(f"ROC-AUC:               {roc_auc:.4f}")
    print(f"Average Precision:     {average_precision}")
    print(f"Run ID:                {score_run_id}")

2026/10/18 12:08:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


XGBoost Score Model Logged!
ROC-AUC:               0.8722
Average Precision:     0.4143803315948079
Run ID:                c31c19db0c6b40129455452fb8f6b7fa


## Register the Model

In [22]:
# Step 1) Register the XGBoost model in Model Registry
from mlflow import register_model

register_model_name = "CreditScoreXGBModel"

model_uri = f"runs:/{score_run_id}/model"

result = register_model(
    model_uri=model_uri,
    name=register_model_name
)

print("Register Model Name: ", result.name)
print("Registered version: ", result.version)

Successfully registered model 'CreditScoreXGBModel'.
2026/10/18 12:28:38 WARNING mlflow.tracking._model_registry.fluent: Run with id c31c19db0c6b40129455452fb8f6b7fa has no artifacts at artifact path 'model', registering model based on models:/m-1af692b4b06b470d98c81763c12743a1 instead


Register Model Name:  CreditScoreXGBModel
Registered version:  1


Created version '1' of model 'CreditScoreXGBModel'.


In [23]:
# Step 2) Saving registered version in a variable
score_model_version = result.version
print("Score model version saved", score_model_version)

Score model version saved 1


In [24]:
# Step 3) Create MLflow Client
from mlflow.client import MlflowClient
client = MlflowClient()
print("MLflow Client is ready!")

MLflow Client is ready!


In [25]:
# Step 4) Move Score Model version to staging
client.transition_model_version_stage(
    name=register_model_name,
    version=score_model_version,
    stage="Staging"
)

<ModelVersion: aliases=[], creation_timestamp=1792351718523, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1792352385872, metrics=None, model_id=None, name='CreditScoreXGBModel', params=None, run_id='c31c19db0c6b40129455452fb8f6b7fa', run_link=None, source='models:/m-1af692b4b06b470d98c81763c12743a1', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [27]:
print(f"Model {register_model_name} version {score_model_version} moved to staging")

Model CreditScoreXGBModel version 1 moved to staging


In [28]:
# Step 5) Check if model qualifies for production
if roc_auc >= 0.75 and average_precision >= 0.25:
    print("Score model is eligible for production")
else:
    print("Score model is not eligible")

Score model is eligible for production


In [30]:
# Step 6) Promote Score model to Production
client.transition_model_version_stage(
    name=register_model_name,
    version=score_model_version,
    stage="Production"
)

print(f"Model {register_model_name} version {score_model_version} is promoted to production!")

Model CreditScoreXGBModel version 1 is promoted to production!


In [35]:
# Step 7) Load the Production Risk Model from Registry
import mlflow.sklearn

production_score_model = mlflow.sklearn.load_model(
    model_uri=f"models:/{register_model_name}/Production"
)

print("Production Risk Model Loaded ✓")

Production Risk Model Loaded ✓


In [36]:
# Step 8) Fetch the current production version
latest_version = client.get_latest_versions(
    name=register_model_name,
    stages=["Production"]
)

production_version = latest_version[0].version if latest_version else None
print("Current Production Version: ", production_version)

Current Production Version:  1


In [38]:
# Step 9) Run prediction by latest version
score_pred = production_score_model.predict_proba(X_test_score.head(5))
print("Predictions", score_pred)

Predictions [[0.95931464 0.04068534]
 [0.56492877 0.4350712 ]
 [0.7293713  0.27062872]
 [0.9916243  0.0083757 ]
 [0.9751248  0.02487524]]


In [39]:
# Step 10) Log Prediction with Model Version
import hashlib
from datetime import datetime

def hash_input(payload: dict) -> str:
    payload_str = json.dumps(payload, sort_keys=True)
    return hashlib.sha256(payload_str.encode()).hexdigest()

In [40]:
# Step 11) Input Payload
input_payload = {
    "RevolvingUtilizationOfUnsecuredLines": 0.56,
    "age": 43,
    "NumberOfTime30-59DaysPastDueNotWorse": 2,
    "DebtRatio": 0.2,
    "MonthlyIncome": 15000,
    "NumberOfOpenCreditLinesAndLoans": 1,
    "NumberOfTimes90DaysLate": 1,
    "NumberRealEstateLoansOrLines": 0,
    "NumberOfTime60-89DaysPastDueNotWorse": 1,
    "NumberOfDependents": 2,
    "total-number_of_lates": 4,
    "log_transformed-montly-income": 0.85
}

In [ ]:
# Step 12) Creating Hash
input_hash = hash_input(input_payload)
print("Input hash: ", input_hash)

Input hash:  3164ae8bcbecb0538e984bf9dff1906fb5298822488da65a361c51cc10824111


In [43]:
# Step 13) Creating Log Record
from datetime import datetime, timezone

BASE_DIR = os.getcwd()  # or project root if running from notebooks
LOG_DIR = os.path.join(BASE_DIR, "logs")

# Step 2: Create logs folder if not exists
os.makedirs(LOG_DIR, exist_ok=True)

# Step 3: Define log file path
LOG_FILE = os.path.join(LOG_DIR, "predictions.log")

# Step 4:
prediction_log = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model_name": register_model_name,
    "model_version": production_version,
    "input_hash": input_hash,
    "prediction": str(score_pred[0])
}

with open(LOG_FILE, "a", encoding="utf-8") as f:
    f.write(json.dumps(prediction_log) + "\n")

print("Prediction logged with model_version ✓")

Prediction logged with model_version ✓
